# 違反ヒートマップ — 「どの非線形制約の緩和が緩いか」を1枚で見る

MINLPの非線形制約はそれぞれ違うタイプの凸緩和(McCormick・べき乗の外側近似・exp凸包絡…)を
持ち、緩さもバラバラ。`minlpkit.collectors.violation` は **ルートLP緩和解を真の非線形制約に
代入**して、どの制約タイプがどれだけ「本来満たすべき式」から外れているかを測る。
これは district_heating_detailed_physics で1モデル分だけ触れた収集器を、
**制約タイプが多いモデル**(反応速度論・転化率・需要充足・除熱能力…6種)で深掘りする。

1. **何を見るか** — `getNlRowSolFeasibility` による違反量の測り方
2. **可視化** — 制約タイプ×エンティティのヒートマップ + タイプ別ランキング
3. **読み解き** — 支配的ボトルネックの特定と `weak_relaxation` 診断ルールとの対応
4. **まとめ**

題材は **バッチ反応器スケジューリング**(`samples/others/scheduling_plant.py`)。
反応速度論(Arrhenius: `exp(1/T)`)・転化率(ネスト`exp`)・需要充足(`n·s·X` 三重積)・
昇温時間(双線形)・除熱制約(三重積相当)と、非線形の種類が豊富。

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
SEQ_BLUE = [[0.0, "#eef5fd"], [0.2, "#cde2fb"], [0.4, "#86b6ef"],
           [0.6, "#3987e5"], [0.8, "#1c5cab"], [1.0, "#0d366b"]]
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 何を見るか — ルートLP緩和解での違反量

`_RootLPViolation`(`Eventhdlr`)は `FIRSTLPSOLVED`(最初のLP求解=ルート)で発火し、
LP緩和解を `Sol` として組み立てたうえで、モデルが保持する各 `NlRow`(非線形緩和行、
元の制約名を保持)に対して

- `getNlRowSolFeasibility(nlrow, sol)` — 負なら違反、その絶対値が違反量
- `getNlRowSolActivity(nlrow, sol)` — 左辺の活動値(正規化に使う)

を測る。制約ごとにスケールが桁違いなので **相対違反 = 違反量 / (|活動値| + 1)** を主指標にする。
`violation_by_type` は制約名を `ctype_entity` の形式(例 `demand_J5`)から `ctype`/`entity` に
分解し、タイプ別に平均・最大・件数を集計する。

In [2]:
vdf = collect_root_violations(sp.build_model())
print(f"収集した非線形制約行: {len(vdf)}本")
vdf[["constraint", "ctype", "entity", "activity", "rel_violation"]].sort_values(
    "rel_violation", ascending=False).head(8)

収集した非線形制約行: 47本


,constraint,ctype,entity,activity,rel_violation
43,demand_J6,demand,J6,43.691535,1.259936
44,batchtime_J6,batchtime,J6,-4.214734,1.095882
30,batchtime_J4,batchtime,J4,-0.546171,1.000000
39,energy_J5,energy,J5,-4907.130644,0.999796
18,energy_J2,energy,J2,-3406.964587,0.999707
32,energy_J4,energy,J4,-2981.531530,0.999665
11,energy_J1,energy,J1,-2404.038587,0.999584
25,energy_J3,energy,J3,-1518.407813,0.999342


## 2. 可視化 — タイプ×エンティティのヒートマップ

行=制約タイプ(平均相対違反の降順)、列=ジョブ/マシン。色が濃いほど緩和が緩い。

In [3]:
by_type = violation_by_type(vdf)
type_order = by_type["ctype"].tolist()
ent_order = sorted(vdf["entity"].unique(), key=lambda e: (e[0] != "J", e))
piv = (vdf.pivot_table(index="ctype", columns="entity", values="rel_violation", aggfunc="max")
      .reindex(index=type_order, columns=ent_order))
z = piv.values
text = [["" if v != v else f"{v:.2f}" for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z, x=ent_order, y=type_order, colorscale=SEQ_BLUE, zmin=0,
    text=text, texttemplate="%{text}", textfont=dict(size=10),
    colorbar=dict(title=dict(text="相対違反", side="right"), thickness=12,
                 tickfont=dict(color=C["muted"], size=10)),
    hovertemplate="%{y}_%{x}<br>相対違反 %{z:.3f}<extra></extra>", xgap=2, ygap=2))
fig.update_layout(
    title=dict(text="ルートLP緩和解の非線形制約違反(相対)— 濃いほど緩和が緩い",
              font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="ジョブ / マシン", side="top", tickfont=dict(color=C["ink2"])),
    yaxis=dict(autorange="reversed", tickfont=dict(color=C["ink2"])),
    margin=dict(l=90, r=20, t=64, b=16), height=420)
show(fig)

In [4]:
g = by_type.iloc[::-1]
fig = go.Figure(go.Bar(
    x=g["mean_rel"], y=g["ctype"], orientation="h", marker=dict(color=C["s1"]),
    text=[f"{v:.2f}" for v in g["mean_rel"]], textposition="outside",
    customdata=g[["max_rel", "n"]],
    hovertemplate="%{y}<br>平均相対違反 %{x:.3f}<br>最大 %{customdata[0]:.3f} / %{customdata[1]}本<extra></extra>"))
fig.update_layout(
    title=dict(text="制約タイプ別の平均相対違反(支配的ボトルネックの特定)",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="平均相対違反", gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"]), zeroline=False),
    yaxis=dict(tickfont=dict(color=C["ink2"])),
    margin=dict(l=90, r=48, t=44, b=40), height=300)
show(fig)

## 3. 読み解き — 診断ルールとの対応

`bottleneck_type`/`bottleneck_rel_viol`(診断が使う集約値)はこのランキングの先頭そのもの。
`weak_relaxation` ルールは「相対違反 ≥ 0.5 **かつ** 空間分枝の寄与 ≥ 0.3」で発火する
(空間分枝の寄与は [notebook 1: McCormick+空間分枝木](01_mccormick_spatial_tree.ipynb) 参照)。

In [5]:
top = by_type.iloc[0]
print(f"支配的ボトルネック: {top['ctype']}  平均相対違反 {top['mean_rel']:.2f}")

report = mk.analyze(sp.build_model, name="plant-violation", time_limit=10)
print(report.summary())
print("report.metrics の bottleneck_type:", report.metrics.get("bottleneck_type"),
     " / bottleneck_rel_viol:", f"{report.metrics.get('bottleneck_rel_viol', 0):.2f}")

支配的ボトルネック: energy  平均相対違反 1.00


[plant-violation] 検出症状 4件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  [good] 制約-変数がブロック構造 + 少数の結合制約 -> ベンダーズ分解 / Dantzig-Wolfe分解(結合制約を主問題に)
report.metrics の bottleneck_type: energy  / bottleneck_rel_viol: 1.00


ヒートマップの先頭タイプと `report.metrics["bottleneck_type"]` が一致することが
確認できる。**energy**(三重積 `n·s·(T−T0)`、目的関数に直結)と **conversion**(ネスト`exp`)が
突出して緩く、**arrhenius**(単変数`exp`)・**jobtime**・**tmax** はタイトなまま残る
——多変数の積が絡む項ほど McCormick 系の緩和が緩くなりやすいという傾向が、複数タイプを
横に並べたことで一目で分かる(1タイプだけの `district_heating` 版では見えない対比)。

## まとめ

- 違反ヒートマップは「ルートLPが真の制約からどれだけ乖離しているか」を制約タイプ×エンティティで
  可視化する、`weak_relaxation` 診断の一次情報そのもの。
- 相対化(`/(|活動値|+1)`)しないと桁違いのスケール(energyは~1e3)に埋もれるため必須。
- 支配的ボトルネックが特定できれば、[手法notebook1: 整数×連続の厳密線形化](../improve/01_linearize_product.ipynb)
  や [手法notebook2: PWL-SOS2](../improve/02_pwl_sos2.ipynb) の適用対象が絞れる。

関連: [プレイブック 0. 診断そのもの](../../playbook/00-diagnose.md) /
API [`minlpkit.collectors`](../../api/pipeline.md)。